# 项目：基于城市声学监测的烟花爆竹识别与告警数据分析

## 分析目标

本数据分析报告的目的是，基于城市声学监测系统中的模拟事件数据，对烟花爆竹识别场景下的样本分布、误报来源、场景差异和阈值权衡进行系统分析，并结合 Python 与 SQL 完成数据清洗、探索分析和指标评估，最终为模型优化和产品策略设计提供依据。

具体而言，本次分析希望回答以下几个问题：

（1）当前数据集中，正负样本的分布情况如何，是否存在类别不平衡问题；

（2）模型的误报主要来源于哪些环境噪声，哪些场景和时间段更容易出现误报；

（3）当前分类效果中，Precision、Recall 和 F1 的表现如何，仅看 Accuracy 是否足以反映业务价值；

（4）当报警阈值发生变化时，Precision、Recall、告警量和人工复核成本如何变化，产品上线时应该如何做指标权衡；

（5）从产品经理视角出发，后续应该优先优化哪些问题，才能提升整套系统的实际可用性。


## 简介

在城市禁燃区和公共安全监管场景中，传统监测方式主要依赖人工巡查、视频监控和红外技术，但这些手段在夜间、逆光、遮挡和复杂环境噪声条件下存在明显局限，往往难以及时发现异常事件，也难以实现快速取证和精准溯源。

本项目围绕“基于声学感知的烟花爆竹识别与定位系统”展开，通过前端麦克风阵列采集环境声音，结合声学分类模型、多点协同定位和云端管理平台，实现异常爆鸣类声音的识别、告警和事件追踪。系统整体采用“端—边—云”协同架构，目标是在复杂城市背景声环境下实现较高精度的识别与较低延迟的事件响应。

由于原始项目数据主要来自音频和视频采集，不适合直接公开展示，因此本案例采用基于真实业务逻辑构造的模拟数据集开展分析。虽然数据为模拟生成，但数据表结构、字段设计、误报类型、告警链路和指标约束都尽量贴近项目真实场景，能够较好地反映产品分析和指标设计过程。


本次分析聚焦于项目中的“声音分类与告警分析”部分，重点展示以下能力：

（1）数据清洗与质量评估能力：能够对原始事件数据进行结构检查、缺失值处理、异常值修正和字段标准化；

（2）探索性数据分析能力：能够通过可视化发现样本分布、误报来源和场景差异；

（3）分类效果评估能力：能够从 Precision、Recall、F1 等多个指标分析当前模型表现，而不是只关注 Accuracy；

（4）业务指标设计能力：能够结合实际监管场景，分析阈值变化对告警质量和人工复核成本的影响；

（5）SQL 查询能力：能够从数据库视角定位高误报场景、重点噪声类型和高风险设备；

（6）产品思考能力：能够把模型问题转化为产品问题，并提出后续可执行的优化方向。


## 读取数据

导入数据分析所需要的库。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)


本案例将使用多张数据表共同完成分析。不同于传统的单表分析任务，本项目的数据来自一条较完整的业务链路，因此数据被组织为多张相互关联的表：

（1）`device_info.csv`：记录设备的部署信息，包括设备编号、部署场景、安装位置和阈值版本等；

（2）`audio_clip_event.csv`：记录音频事件片段，是本次分析最核心的主表，包括事件时间、设备编号、场景、天气、真实类别、模型分数和分类结果等字段；

（3）`alert_record.csv`：记录模型触发告警后的结果，包括告警分数、是否推送到平台、人工复核结果等；

（4）`firework_event_case.csv`：记录多个音频片段或多个设备融合后的事件级记录，用于表示最终业务事件；

（5）`threshold_experiment_log.csv`：记录不同阈值下的分类效果和成本结果，用于后续阈值分析；

（6）`fireworks_monitoring_cleaned.db`：将以上数据整合后的 SQLite 数据库，用于 SQL 查询分析。


首先读取原始音频事件表，并查看头部数据。


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_AUDIO = DATA_DIR / 'audio_clip_event.csv'
RAW_DB = DATA_DIR / 'fireworks_monitoring.db'
CLEAN_AUDIO = DATA_DIR / 'audio_clip_event_cleaned.csv'
CLEAN_DB = DATA_DIR / 'fireworks_monitoring_cleaned.db'

original_audio_event = pd.read_csv(RAW_AUDIO, parse_dates=['event_time'])
original_audio_event.head()


为了区分原始数据和后续清洗后的数据，我们创建一个副本 `cleaned_audio_event`，后续所有清洗和整理操作都基于该副本进行。


In [ ]:
cleaned_audio_event = original_audio_event.copy()


## 评估和清理数据

在这一部分中，我们将对原始音频事件数据进行评估和清理。

主要从两个方面进行：

（1）数据整齐度：查看数据是否符合“每个变量为一列，每个观察值为一行”的基本要求；

（2）数据干净度：检查是否存在缺失值、重复值、异常值、命名不一致和无效数据等问题。


### 数据整齐度
先查看数据头部内容，判断数据的基本结构。


In [ ]:
cleaned_audio_event.head(10)


从头部数据来看，当前数据基本符合“每个变量为一列、每个事件为一行”的要求，因此不存在明显的结构性问题。后续主要关注内容层面的清理。


### 数据干净度
接下来通过 `info()` 和缺失值统计，对数据内容进行整体了解。


In [ ]:
cleaned_audio_event.info()

print('')
print('缺失值统计：')
print(cleaned_audio_event.isna().sum().sort_values(ascending=False).head(15))

print('')
print('时间范围：')
print(cleaned_audio_event['event_time'].min(), 'to', cleaned_audio_event['event_time'].max())

print('')
print('重复 clip_id 检查：', int(cleaned_audio_event['clip_id'].duplicated().sum()))


从输出结果来看，当前原始数据主要存在以下几类问题：

（1）部分字段存在缺失值，尤其是天气字段；

（2）场景类型字段存在命名不一致的问题，例如同一场景可能出现多个不同写法；

（3）存在重复的音频片段记录；

（4）部分数值字段可能超出合理范围，例如模型分数应位于 0 到 1 之间，峰值分贝和信噪比也应处于现实可解释范围内。

因此，后续需要对这些问题进行统一处理。


### 处理命名不一致问题

在场景字段中，同一场景可能存在不同写法。为了保证后续统计结果准确，需要先对这些命名进行标准化。


In [ ]:
scene_map = {
    'residential': 'residential_area',
    'residental_area': 'residential_area',
    'main_road': 'arterial_road',
    'arterialRoad': 'arterial_road',
    'construction': 'construction_site',
    'site_construction': 'construction_site',
    'industrialPark': 'industrial_park',
    'industry_park': 'industrial_park',
}
cleaned_audio_event['scene_type'] = cleaned_audio_event['scene_type'].replace(scene_map)


### 处理缺失值

对于缺失的天气字段，如果直接删除会导致部分事件样本丢失。由于天气字段主要用于场景解释而不是模型训练，因此这里采用统一填补的方式进行处理，以保留更多可分析样本。


In [ ]:
mode_by_group = (
    cleaned_audio_event.dropna(subset=['weather'])
    .groupby(['scene_type', 'time_bucket'])['weather']
    .agg(lambda x: x.mode().iloc[0])
)

def fill_weather(row):
    if pd.notna(row['weather']):
        return row['weather']
    key = (row['scene_type'], row['time_bucket'])
    if key in mode_by_group.index:
        return mode_by_group[key]
    return 'cloudy'

cleaned_audio_event['weather'] = cleaned_audio_event.apply(fill_weather, axis=1)


### 处理重复值

音频片段的唯一标识符 `clip_id` 不应重复，因此需要对重复记录进行去重处理。


In [ ]:
before_drop_dup = len(cleaned_audio_event)
cleaned_audio_event = cleaned_audio_event.drop_duplicates(subset=['clip_id'], keep='first').copy()
removed_dup = before_drop_dup - len(cleaned_audio_event)
print('已删除重复记录数:', removed_dup)


### 处理异常值

从业务逻辑看，模型分数应位于 0 到 1 之间，峰值分贝和信噪比也应处于合理范围。对于明显超出常理的数据，需要进行裁剪或修正，以避免影响后续分析结果。


In [ ]:
before_peak_filter = len(cleaned_audio_event)
cleaned_audio_event = cleaned_audio_event[
    (cleaned_audio_event['peak_db'] >= 25) & (cleaned_audio_event['peak_db'] <= 140)
].copy()
removed_peak_outliers = before_peak_filter - len(cleaned_audio_event)

cleaned_audio_event['model_score'] = cleaned_audio_event['model_score'].clip(0.0, 1.0)
cleaned_audio_event['snr_db'] = cleaned_audio_event['snr_db'].clip(-15, 50)
cleaned_audio_event['duration_ms'] = cleaned_audio_event['duration_ms'].clip(50, 5000)
cleaned_audio_event['spectral_centroid'] = cleaned_audio_event['spectral_centroid'].clip(200, 8000)
cleaned_audio_event['zero_crossing_rate'] = cleaned_audio_event['zero_crossing_rate'].clip(0.0, 1.0)
cleaned_audio_event['rms_energy'] = cleaned_audio_event['rms_energy'].clip(0.0, 1.0)

print('已移除不合理 peak_db 记录数:', removed_peak_outliers)


### 清洗结果检查

完成清洗后，再次检查缺失值、重复值和场景字段分布，确认数据质量是否满足后续分析要求。


In [ ]:
print('weather 缺失值:', int(cleaned_audio_event['weather'].isna().sum()))
print('重复 clip_id:', int(cleaned_audio_event['clip_id'].duplicated().sum()))

print('')
print('场景分布（清洗后）：')
print(cleaned_audio_event['scene_type'].value_counts())

print('')
print('标签分布（清洗后）：')
print(cleaned_audio_event['binary_label'].value_counts().sort_index())

# 基于清洗后的数据直接计算混淆矩阵（不使用预设目标值）
y_true_clean = cleaned_audio_event['binary_label'].astype(int)
y_pred_clean = cleaned_audio_event['predicted_label'].astype(int)

tn = int(((y_true_clean == 0) & (y_pred_clean == 0)).sum())
fp = int(((y_true_clean == 0) & (y_pred_clean == 1)).sum())
fn = int(((y_true_clean == 1) & (y_pred_clean == 0)).sum())
tp = int(((y_true_clean == 1) & (y_pred_clean == 1)).sum())

confusion_from_data = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=['真实: 非烟花(0)', '真实: 烟花(1)'],
    columns=['预测: 非烟花(0)', '预测: 烟花(1)'],
)

print('')
print('混淆矩阵（基于当前数据计算）：')
print(confusion_from_data)

print('')
print('TP =', tp, 'FP =', fp, 'FN =', fn, 'TN =', tn)


从清洗结果来看：

（1）天气缺失值已经完成填补；

（2）重复片段记录已被移除；

（3）场景类型字段已经统一到标准命名；

（4）关键数值字段已经回到合理范围内；

（5）混淆矩阵已基于清洗后数据直接计算，后续指标分析不再依赖预设值。

因此，当前清洗后的数据可以作为后续探索分析和指标分析的基础。


## 整理数据

在正式进行探索性分析前，还需要对清洗后的数据做进一步整理，以便更好地服务后续统计和可视化。

本项目中，除了原始字段外，我们还重点关注以下几个派生角度：

（1）按 `binary_label` 区分正样本和负样本；

（2）按 `error_type` 区分 TP、FP、FN、TN 四类结果；

（3）按 `scene_type` 分析不同部署场景下的误报表现；

（4）按 `time_bucket` 分析不同时段的误报情况；

（5）按 `model_score` 分析不同错误类型的置信度分布。

这些整理后的字段可以帮助我们从“样本分布—误报归因—阈值设计”这条主线展开完整分析。


In [ ]:
analysis_event = cleaned_audio_event.copy()
analysis_event.head()


## 探索数据

在着手分析分类效果和阈值问题之前，我们先通过可视化方式观察数据的整体分布，重点关注以下几个问题：

（1）当前数据集中正负样本的比例如何；

（2）负样本中不同噪声类型的分布情况如何；

（3）误报主要来自哪些噪声类型；

（4）不同场景和不同时段下的误报率是否存在明显差异；

（5）TP、FP、FN、TN 在模型分数上的分布是否有明显区别。

这些分析可以帮助我们快速定位后续的重点问题。


### 样本分布
首先查看正样本和负样本的分布情况。


In [ ]:
FIG_DIR = PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

label_counts = analysis_event['binary_label'].value_counts().sort_index()
labels = ['Negative (0)', 'Positive (1)']
values = [int(label_counts.get(0, 0)), int(label_counts.get(1, 0))]

plt.figure(figsize=(7, 4))
plt.bar(labels, values, color=['#6b7280', '#ef4444'])
for i, v in enumerate(values):
    plt.text(i, v + 50, str(v), ha='center')
plt.title('Sample Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'sample_distribution.png', dpi=140)
plt.show()


![样本分布图](figures/sample_distribution.png)

从样本分布图来看，负样本数量明显高于正样本，说明当前任务存在较明显的类别不平衡问题。

这意味着后续在评估模型效果时，不能只看整体准确率。因为在负样本占比较高的情况下，即使模型对多数负样本判断较准，也可能掩盖误报和漏报在业务层面的真实影响。


### 负样本噪声类型分布
接下来查看负样本内部的噪声类型构成。


In [ ]:
neg_event = analysis_event[analysis_event['binary_label'] == 0]
noise_dist = neg_event['true_class'].value_counts()

plt.figure(figsize=(9, 4))
noise_dist.plot(kind='bar', color='#60a5fa')
plt.title('Noise Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'noise_class_distribution.png', dpi=140)
plt.show()


![负样本噪声类型分布图](figures/noise_class_distribution.png)

从负样本类型分布来看，环境噪声主要由交通噪声、施工噪声、工业噪声等类别构成。其中，重型货车气喇叭和工地打桩声占比相对较高。

这说明当前识别任务并不是简单地区分“烟花”和“非烟花”，而是在多个高相似度噪声之间做精细区分，因此后续误报分析尤其重要。


### 误报来源分析
仅观察 FP 样本的真实类别分布，定位当前模型最容易混淆的噪声来源。


In [ ]:
fp_event = analysis_event[analysis_event['error_type'] == 'FP']
fp_root = fp_event['true_class'].value_counts()

plt.figure(figsize=(9, 4))
fp_root.plot(kind='bar', color='#f59e0b')
plt.title('FP Root Cause Distribution')
plt.ylabel('FP Count')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fp_root_cause.png', dpi=140)
plt.show()


![误报来源柱状图](figures/fp_root_cause.png)

从误报来源图来看，误报主要集中在重型货车气喇叭和工地打桩声。这两类声音都具备高分贝、瞬时性或脉冲性较强的特点，在声纹层面与烟花爆竹的爆鸣声存在一定相似性，因此更容易触发误报。

从产品角度看，这类误报会直接影响系统告警的可信度，并增加人工复核和无效出警成本，因此应作为后续优化的优先方向。


### 不同场景下的误报率
进一步比较不同部署场景中的误报率差异。


In [ ]:
scene_stat = analysis_event.groupby('scene_type', as_index=False).agg(
    total_samples=('clip_id', 'count'),
    fp_count=('error_type', lambda x: int((x == 'FP').sum())),
)
scene_stat['fp_rate'] = scene_stat['fp_count'] / scene_stat['total_samples']
scene_stat = scene_stat.sort_values('fp_rate', ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(scene_stat['scene_type'], scene_stat['fp_rate'], color='#34d399')
plt.title('FP Rate by Scene')
plt.ylabel('FP Rate')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'scene_fp_rate.png', dpi=140)
plt.show()

scene_stat


![按场景误报率图](figures/scene_fp_rate.png)

从不同场景误报率对比来看，工地周边和主干道场景的误报率通常更高，而居民区和工业园相对更稳定。

这说明误报并不只是模型本身的问题，还与部署场景的噪声结构密切相关。后续如果进入真实上线阶段，可以考虑按场景做差异化阈值策略，而不是对所有设备统一采用同一阈值。


### 不同时段下的误报率
继续比较不同时间段的误报率差异。


In [ ]:
bucket_order = ['daytime', 'evening_peak', 'night', 'early_morning']
bucket_stat = analysis_event.groupby('time_bucket', as_index=False).agg(
    total_samples=('clip_id', 'count'),
    fp_count=('error_type', lambda x: int((x == 'FP').sum())),
)
bucket_stat['fp_rate'] = bucket_stat['fp_count'] / bucket_stat['total_samples']
bucket_stat['time_bucket'] = pd.Categorical(bucket_stat['time_bucket'], categories=bucket_order, ordered=True)
bucket_stat = bucket_stat.sort_values('time_bucket')

plt.figure(figsize=(8, 4))
plt.bar(bucket_stat['time_bucket'].astype(str), bucket_stat['fp_rate'], color='#a78bfa')
plt.title('FP Rate by Time Bucket')
plt.ylabel('FP Rate')
plt.tight_layout()
plt.savefig(FIG_DIR / 'time_bucket_fp_rate.png', dpi=140)
plt.show()

bucket_stat


![按时段误报率图](figures/time_bucket_fp_rate.png)

从不同时段误报率来看，部分时间段的误报问题更明显。一般而言，夜间主干道上的高分贝车辆噪声，以及白天工地的持续施工噪声，都是误报高发的重要来源。

这表明时段因素对模型表现也有影响，后续可以结合场景和时间段做更细粒度的风险分层。


### 模型分数分布
最后观察 TP、FP、FN、TN 四类结果在模型分数上的分布情况。


In [ ]:
plt.figure(figsize=(9, 5))
for err, color in [('TP', '#22c55e'), ('FP', '#ef4444'), ('FN', '#f97316'), ('TN', '#3b82f6')]:
    vals = analysis_event.loc[analysis_event['error_type'] == err, 'model_score']
    plt.hist(vals, bins=35, alpha=0.45, density=True, label=err, color=color)

plt.axvline(0.80, color='black', linestyle='--', linewidth=1, label='threshold=0.8')
plt.title('Score Distribution by Error Type')
plt.xlabel('Model Score')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'score_distribution.png', dpi=140)
plt.show()


![分数分布图](figures/score_distribution.png)

从分数分布来看，TP 和 TN 在大多数情况下分布较为分离，但 FP 和 FN 会更多地集中在阈值附近。这说明模型在边界样本上的判断仍存在较大不确定性。

这一发现对后续阈值分析非常关键，因为它意味着适度上调或下调阈值，会直接影响这部分边界样本的判定结果，从而改变 Precision 和 Recall 的平衡关系。


## 分析数据：分类效果与阈值权衡

在探索性分析之后，接下来进入更偏指标评估的部分。这里不再只观察数据分布，而是从分类任务本身出发，回答以下两个问题：

（1）当前模型在默认阈值下的 Precision、Recall、F1 和 Accuracy 表现如何；

（2）如果调整报警阈值，这些指标以及告警量、误报成本会如何变化。

对于实际产品来说，这一部分比单纯的 Accuracy 更重要，因为业务系统真正关心的是“告警是否可信”以及“误报是否可控”。


### 默认阈值下的分类效果

首先查看默认阈值（0.80）下的分类结果。


In [ ]:
# 先基于当前数据中的真实标签和预测标签直接计算混淆矩阵
# 这里不预设 TP/FP/FN/TN 的目标值，而是由数据本身决定
y_true_data = analysis_event['binary_label'].astype(int).to_numpy()
y_pred_data = analysis_event['predicted_label'].astype(int).to_numpy()

tn_data = int(((y_true_data == 0) & (y_pred_data == 0)).sum())
fp_data = int(((y_true_data == 0) & (y_pred_data == 1)).sum())
fn_data = int(((y_true_data == 1) & (y_pred_data == 0)).sum())
tp_data = int(((y_true_data == 1) & (y_pred_data == 1)).sum())

confusion_table = pd.DataFrame(
    [[tn_data, fp_data], [fn_data, tp_data]],
    index=['真实: 非烟花(0)', '真实: 烟花(1)'],
    columns=['预测: 非烟花(0)', '预测: 烟花(1)'],
)

print('数据驱动混淆矩阵：')
print(confusion_table)
print('TP =', tp_data, 'FP =', fp_data, 'FN =', fn_data, 'TN =', tn_data)

# 阈值评估函数
# 用于评估不同阈值变化下的 Precision / Recall / F1 与成本
y_true = analysis_event['binary_label'].astype(int).to_numpy()
score = analysis_event['model_score'].to_numpy()

def metrics_at_threshold(y_true, score, threshold):
    pred = (score >= threshold).astype(int)
    tp = int(((y_true == 1) & (pred == 1)).sum())
    fp = int(((y_true == 0) & (pred == 1)).sum())
    fn = int(((y_true == 1) & (pred == 0)).sum())
    tn = int(((y_true == 0) & (pred == 0)).sum())

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    review_cost = (tp + fp) * 8
    false_alarm_cost = fp * 80
    miss_cost = fn * 300
    total_cost = review_cost + false_alarm_cost + miss_cost

    return {
        'threshold': threshold,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'alert_volume': tp + fp,
        'review_cost': review_cost,
        'false_alarm_cost': false_alarm_cost,
        'miss_cost': miss_cost,
        'estimated_total_cost': total_cost,
    }

baseline = pd.DataFrame([metrics_at_threshold(y_true, score, 0.80)])
baseline[['threshold', 'tp', 'fp', 'fn', 'tn', 'accuracy', 'precision', 'recall', 'f1']]


从默认阈值下的结果来看，当前模型在整体上已经具备较好的分类能力，但如果从业务角度出发，仅看 Accuracy 仍然是不够的。

在监管或告警系统中，误报会直接带来人工复核、无效出警和用户信任下降等问题，因此 Precision 往往比单纯的 Accuracy 更能体现产品价值。而 Recall 反映的是漏检风险，两者之间需要结合场景做平衡。


本报告中的 TP、FP、FN、TN 由当前数据直接计算得到，而不是预先设定目标混淆矩阵。


### PR 曲线分析

为了更直观地观察 Precision 和 Recall 的关系，接下来绘制 PR 曲线。


In [ ]:
sweep_threshold = np.linspace(0.0, 1.0, 201)
pr_df = pd.DataFrame([metrics_at_threshold(y_true, score, t) for t in sweep_threshold])

plt.figure(figsize=(6, 5))
plt.plot(pr_df['recall'], pr_df['precision'], color='#2563eb', linewidth=2)
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pr_curve.png', dpi=150)
plt.show()


![PR 曲线](figures/pr_curve.png)

从 PR 曲线可以看出，随着阈值变化，Precision 和 Recall 之间会出现明显的此消彼长关系。

如果阈值较低，模型会更容易触发告警，Recall 较高，但 Precision 往往下降；如果阈值上调，Precision 会提升，但 Recall 会有所牺牲。因此，阈值设计本质上是一次业务权衡，而不是单纯的技术参数调整。


### 阈值变化下的指标权衡

接下来进一步观察不同阈值下 Precision 和 Recall 的变化趋势。


In [ ]:
threshold_grid = np.linspace(0.50, 0.95, 91)
tradeoff_df = pd.DataFrame([metrics_at_threshold(y_true, score, t) for t in threshold_grid])

plt.figure(figsize=(8, 4.5))
plt.plot(tradeoff_df['threshold'], tradeoff_df['precision'], label='Precision', color='#16a34a', linewidth=2)
plt.plot(tradeoff_df['threshold'], tradeoff_df['recall'], label='Recall', color='#dc2626', linewidth=2)
plt.title('Threshold vs Precision / Recall')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'threshold_tradeoff.png', dpi=150)
plt.show()


![阈值权衡图](figures/threshold_tradeoff.png)

从阈值变化图可以看出，当阈值逐步提高时，Precision 呈上升趋势，而 Recall 则逐步下降。这说明系统正在变得“更保守”，只有更高置信度的样本才会被判定为目标事件。

对于本项目所对应的城市禁燃监管场景而言，这种趋势是符合业务预期的。因为相比于偶尔漏掉少量低风险事件，业务上通常更难接受的是频繁误报导致的无效出警和平台告警失真。


### 不同阈值下的成本估算

除了观察指标变化外，还需要结合人工复核和误报成本做进一步判断。


In [ ]:
compare_thresholds = [0.60, 0.70, 0.80, 0.85, 0.90]
cost_table = pd.DataFrame([metrics_at_threshold(y_true, score, t) for t in compare_thresholds])
cost_table = cost_table[[
    'threshold', 'precision', 'recall', 'f1', 'alert_volume', 'fp', 'fn',
    'review_cost', 'false_alarm_cost', 'miss_cost', 'estimated_total_cost'
]]

candidate_086 = pd.DataFrame([metrics_at_threshold(y_true, score, 0.86)])[ [
    'threshold', 'precision', 'recall', 'f1', 'alert_volume', 'fp', 'fn', 'estimated_total_cost'
] ]

print('不同阈值成本对比：')
cost_table
print('')
print('阈值 0.86 参考：')
candidate_086


从成本估算结果来看，阈值并不是越低越好，也不是越高越好。阈值过低会带来大量误报，增加人工复核和出警成本；阈值过高虽然能提升 Precision，但也可能让 Recall 下降过多，从而漏掉真正的重要事件。

因此，最合理的做法是结合具体业务目标选择阈值。例如，在强调告警可信度的场景下，可以适当上调阈值；在强调事件覆盖率的场景下，则可以适度保留更高 Recall。


## SQL 查询分析

除了使用 Pandas 和 Matplotlib 进行数据分析外，本项目还通过 SQLite 数据库完成了 SQL 查询分析。这一步的意义在于：很多真实业务场景中的分析并不是直接在 DataFrame 中完成的，而是需要通过数据库查询快速定位问题。

在这一部分中，我将通过几条典型 SQL 语句，分别查看：

（1）不同场景下的误报率；

（2）误报来源的主要类别；

（3）不同阈值版本下的告警表现。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path

SQL_DB = DATA_DIR / 'fireworks_monitoring_cleaned.db'
if not SQL_DB.exists():
    SQL_DB = DATA_DIR / 'fireworks_monitoring.db'

conn = sqlite3.connect(SQL_DB)


def run_sql(sql):
    return pd.read_sql_query(sql, conn)


### 不同场景下的误报率
先通过 SQL 查询不同部署场景的样本量和误报率。


In [ ]:
sql_scene_fp = """
SELECT
    scene_type,
    COUNT(*) AS total_samples,
    SUM(CASE WHEN error_type = 'FP' THEN 1 ELSE 0 END) AS fp_count,
    ROUND(1.0 * SUM(CASE WHEN error_type = 'FP' THEN 1 ELSE 0 END) / COUNT(*), 4) AS fp_rate
FROM audio_clip_event
GROUP BY scene_type
ORDER BY fp_rate DESC;
"""
run_sql(sql_scene_fp)


该查询结果显示，不同场景的误报率存在明显差异。工地和主干道场景通常更容易出现误报，与前面的探索性分析结论一致。

这说明误报问题具有明确的场景属性，因此产品策略不应只停留在“全局统一阈值”层面，而可以进一步考虑按场景配置不同的阈值或复核策略。


### 误报来源 Top 类别
接下来查询 FP 样本中最主要的噪声类型。


In [ ]:
sql_fp_source = """
SELECT
    true_class,
    COUNT(*) AS fp_count
FROM audio_clip_event
WHERE error_type = 'FP'
GROUP BY true_class
ORDER BY fp_count DESC;
"""
run_sql(sql_fp_source)


从 SQL 查询结果可以进一步确认，误报主要来自重型货车气喇叭、工地打桩声等类别。这与前面的图表结果互相印证。

从分析方法上看，这一步说明问题并不依赖某一种工具。无论是用可视化还是用 SQL 查询，只要分析口径一致，都应指向相同的核心结论。


### 不同阈值版本下的告警表现
最后比较不同阈值版本下的平均告警分数、告警量和误报情况。


In [ ]:
sql_threshold_version = """
SELECT
    threshold_version,
    ROUND(AVG(score_at_alert), 4) AS avg_alert_score,
    COUNT(*) AS alert_volume,
    SUM(CASE WHEN operator_review_result = 'false_alarm' THEN 1 ELSE 0 END) AS false_alarm_count
FROM alert_record
GROUP BY threshold_version;
"""
result_threshold = run_sql(sql_threshold_version)
conn.close()
result_threshold


从不同阈值版本的对比结果来看，阈值策略会直接影响告警数量和误报情况。更高的阈值版本通常意味着更少的告警量和更高的告警可信度，但也可能伴随部分召回损失。

这说明阈值并不是单纯的模型参数，而是产品指标体系中的核心配置项。产品经理需要根据实际业务场景定义“更适合上线的阈值”，而不是只追求实验室中的最佳单项指标。


## 结论与产品思考

通过以上的数据清洗、探索分析、指标评估和 SQL 查询，本次分析得出了以下几个核心结论。


### 关键发现

（1）当前数据集存在明显的类别不平衡问题，负样本远高于正样本，因此在评估模型效果时不能只看整体 Accuracy；

（补充）本报告中的混淆矩阵（TP/FP/FN/TN）由最终清洗数据直接计算得到，不采用预设目标值；

（2）误报主要集中在重型货车气喇叭和工地打桩声，这类高分贝、突发性、声纹相近的环境噪声是当前模型最主要的干扰来源；

（3）工地和主干道场景的误报率明显更高，说明误报问题具有明显的场景属性；

（4）模型的边界样本更多地分布在阈值附近，因此阈值调整会显著影响 Precision 和 Recall 的平衡；

（5）从业务角度看，系统上线时更需要关注告警可信度和人工复核成本，因此 Precision 往往比单纯的 Accuracy 更接近实际产品价值。


### 产品侧优化建议

基于本次分析，我认为后续可以从以下几个方向推进产品优化：

（1）优先优化对重型货车气喇叭和工地打桩声的区分能力，降低主要误报来源；

（2）根据不同场景和时间段设计更细粒度的阈值策略，而不是所有设备统一使用同一阈值；

（3）在平台侧加入更精细的复核机制，例如根据分数区间对告警做分级处理，以减少人工压力；

（4）持续补充复杂噪声场景下的数据，提升模型对边界样本和高相似度噪声的鲁棒性。


### 项目总结

这次分析不仅完成了对烟花爆竹声学监测数据的清洗、探索和指标评估，也展示了我在 AI 产品场景中的完整分析能力：能够从数据结构设计、数据质量检查、误报归因、阈值权衡到 SQL 查询分析，逐步定位问题并输出产品决策建议。

对我而言，这个项目的价值并不只是“做出一个分类模型”，而是能够把模型问题转化为产品问题，把实验结果转化为业务指标，并最终形成一套更接近真实落地场景的产品化思考。
